In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
import numpy as np
import os

base_path = '/content/drive/MyDrive/SpatialAI'

Mounted at /content/drive


In [2]:
# Detections from Phase 1
with open(os.path.join(base_path, 'outputs/detections/detections.json'), 'r') as f:
    detections = json.load(f)

# Camera poses from Phase 3 (re-parsing the same way we did before)
def read_images_txt(path):
    cameras = []
    with open(path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        if line.startswith('#') or line.strip() == '':
            continue
        parts = line.split()
        if len(parts) >= 10:
            try:
                image_id = int(parts[0])
                qw, qx, qy, qz = map(float, parts[1:5])
                tx, ty, tz = map(float, parts[5:8])
                name = parts[9]
                cameras.append({'name': name, 'quat': [qw, qx, qy, qz], 'trans': [tx, ty, tz]})
            except ValueError:
                continue
    return cameras

camera_poses = read_images_txt(os.path.join(base_path, 'outputs/camera_poses/sparse_text/images.txt'))
print(f"Loaded detections for {len(detections)} frames")
print(f"Loaded {len(camera_poses)} camera poses")

Loaded detections for 71 frames
Loaded 36 camera poses


In [3]:
def read_cameras_txt(path):
    cameras = {}
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            parts = line.split()
            camera_id = int(parts[0])
            model = parts[1]
            width, height = int(parts[2]), int(parts[3])
            params = list(map(float, parts[4:]))
            cameras[camera_id] = {'model': model, 'width': width, 'height': height, 'params': params}
    return cameras

cameras_intrinsics = read_cameras_txt(os.path.join(base_path, 'outputs/camera_poses/sparse_text/cameras.txt'))
print(cameras_intrinsics)

{1: {'model': 'SIMPLE_RADIAL', 'width': 478, 'height': 850, 'params': [808.2347104358404, 239.0, 425.0, 0.03420700153496236]}}


In [4]:
depth_dir = os.path.join(base_path, 'outputs/depth_maps/raw_arrays')

def load_depth_map(frame_name):
    base_name = os.path.splitext(frame_name)[0]
    return np.load(os.path.join(depth_dir, f"{base_name}.npy"))

# Load sparse 3D points from Phase 3 (already parsed function from before)
def read_points3D_txt(path):
    points = []
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            parts = line.split()
            x, y, z = map(float, parts[1:4])
            points.append([x, y, z])
    return np.array(points)

sparse_points = read_points3D_txt(os.path.join(base_path, 'outputs/camera_poses/sparse_text/points3D.txt'))
print(f"Loaded {len(sparse_points)} sparse 3D points for scale reference")

Loaded 5234 sparse 3D points for scale reference


In [5]:
def quaternion_to_rotation_matrix(qw, qx, qy, qz):
    return np.array([
        [1 - 2*qy**2 - 2*qz**2, 2*qx*qy - 2*qz*qw, 2*qx*qz + 2*qy*qw],
        [2*qx*qy + 2*qz*qw, 1 - 2*qx**2 - 2*qz**2, 2*qy*qz - 2*qx*qw],
        [2*qx*qz - 2*qy*qw, 2*qy*qz + 2*qx*qw, 1 - 2*qx**2 - 2*qy**2]
    ])

f = cameras_intrinsics[1]['params'][0]
cx = cameras_intrinsics[1]['params'][1]
cy = cameras_intrinsics[1]['params'][2]

scale_ratios = []

for cam in camera_poses:
    qw, qx, qy, qz = cam['quat']
    tx, ty, tz = cam['trans']
    R = quaternion_to_rotation_matrix(qw, qx, qy, qz)
    t = np.array([tx, ty, tz])

    depth_map = load_depth_map(cam['name'])

    # Project each sparse 3D point into this camera's view
    for point in sparse_points:
        point_cam = R @ point + t  # transform world point into camera coordinate frame
        if point_cam[2] <= 0:
            continue  # behind the camera, skip

        # Project to pixel coordinates
        u = f * point_cam[0] / point_cam[2] + cx
        v = f * point_cam[1] / point_cam[2] + cy

        if 0 <= int(v) < depth_map.shape[0] and 0 <= int(u) < depth_map.shape[1]:
            colmap_depth = point_cam[2]  # true depth in COLMAP's scale
            midas_depth = depth_map[int(v), int(u)]  # MiDaS's predicted depth at that pixel

            if midas_depth > 0:
                scale_ratios.append(colmap_depth / midas_depth)

scale_factor = np.median(scale_ratios)
print(f"Computed scale factor from {len(scale_ratios)} point observations: {scale_factor:.4f}")

Computed scale factor from 138677 point observations: 0.0151


In [6]:
scale_ratios_arr = np.array(scale_ratios)
print(f"Mean: {scale_ratios_arr.mean():.4f}")
print(f"Median: {np.median(scale_ratios_arr):.4f}")
print(f"Std dev: {scale_ratios_arr.std():.4f}")
print(f"25th percentile: {np.percentile(scale_ratios_arr, 25):.4f}")
print(f"75th percentile: {np.percentile(scale_ratios_arr, 75):.4f}")

Mean: 0.0160
Median: 0.0151
Std dev: 0.0075
25th percentile: 0.0094
75th percentile: 0.0210


In [7]:
camera_by_name = {cam['name']: cam for cam in camera_poses}

In [8]:
def unproject_pixel_to_3d(u, v, depth_value, R, t, f, cx, cy, scale_factor):
    # Scale MiDaS depth into COLMAP's coordinate scale
    depth_scaled = depth_value * scale_factor

    # Reverse the pinhole projection: pixel -> camera-space 3D point
    x_cam = (u - cx) * depth_scaled / f
    y_cam = (v - cy) * depth_scaled / f
    z_cam = depth_scaled
    point_cam = np.array([x_cam, y_cam, z_cam])

    # Camera-space point -> world-space point (undo the world-to-camera transform)
    point_world = R.T @ (point_cam - t)
    return point_world

In [9]:
scene_objects = []

for frame_name, frame_detections in detections.items():
    if frame_name not in camera_by_name:
        continue

    cam = camera_by_name[frame_name]
    qw, qx, qy, qz = cam['quat']
    tx, ty, tz = cam['trans']
    R = quaternion_to_rotation_matrix(qw, qx, qy, qz)
    t = np.array([tx, ty, tz])

    depth_map = load_depth_map(frame_name)

    for det in frame_detections:
        x1, y1, x2, y2 = det['box']
        u = (x1 + x2) / 2  # box center pixel
        v = (y1 + y2) / 2

        if 0 <= int(v) < depth_map.shape[0] and 0 <= int(u) < depth_map.shape[1]:
            depth_value = depth_map[int(v), int(u)]
            if depth_value > 0:
                pos_3d = unproject_pixel_to_3d(u, v, depth_value, R, t, f, cx, cy, scale_factor)
                scene_objects.append({
                    'frame': frame_name,
                    'class': det['class'],
                    'confidence': det['confidence'],
                    'position_3d': pos_3d.tolist()
                })

print(f"Unprojected {len(scene_objects)} object observations into 3D space")

Unprojected 198 object observations into 3D space


In [10]:
from collections import defaultdict

by_class = defaultdict(list)
for obj in scene_objects:
    by_class[obj['class']].append(obj['position_3d'])

for cls, positions in by_class.items():
    positions_arr = np.array(positions)
    print(f"\n{cls} ({len(positions)} observations):")
    print(f"  Mean position: {positions_arr.mean(axis=0)}")
    print(f"  Std dev: {positions_arr.std(axis=0)}")


laptop (36 observations):
  Mean position: [-0.262687    0.36893879  7.662247  ]
  Std dev: [0.58743686 0.11967334 0.78916314]

chair (31 observations):
  Mean position: [-1.87109227  4.11393047 11.25509554]
  Std dev: [0.7139944  1.20814141 3.20118136]

book (105 observations):
  Mean position: [-1.05179632 -0.17437526  6.74488443]
  Std dev: [1.14046038 1.23948145 1.70278187]

keyboard (4 observations):
  Mean position: [0.36176578 1.05387481 9.16689577]
  Std dev: [0.75058526 0.15189296 1.33752772]

cup (18 observations):
  Mean position: [1.25595383 0.03893438 8.50916041]
  Std dev: [0.73753061 0.16345083 1.79207704]

bottle (4 observations):
  Mean position: [ 1.25148111 -1.12403781  6.5664561 ]
  Std dev: [0.24405296 1.09620021 0.55525189]


In [11]:
def cluster_object_class(positions, confidences, class_name):
    positions_arr = np.array(positions)
    confidences_arr = np.array(confidences)

    # Step 1: robust centroid using median (less sensitive to outliers than mean)
    median_pos = np.median(positions_arr, axis=0)

    # Step 2: compute distance of each observation from the median centroid
    distances_from_median = np.linalg.norm(positions_arr - median_pos, axis=1)

    # Step 3: filter out observations that are unusually far from the median (likely outliers/misclassifications)
    # Using median absolute deviation (MAD) - a robust way to define "unusually far"
    mad = np.median(np.abs(distances_from_median - np.median(distances_from_median)))
    threshold = np.median(distances_from_median) + 3 * mad if mad > 0 else np.inf

    inlier_mask = distances_from_median <= threshold
    filtered_positions = positions_arr[inlier_mask]
    filtered_confidences = confidences_arr[inlier_mask]

    # Step 4: recompute a clean centroid from inliers only
    final_position = filtered_positions.mean(axis=0)
    final_std = filtered_positions.std(axis=0)
    spread_magnitude = np.linalg.norm(final_std)  # single number summarizing overall spread

    # Step 5: assign a reliability label based on spread and how many outliers we removed
    n_removed = len(positions_arr) - len(filtered_positions)
    if spread_magnitude < 0.5 and n_removed == 0:
        reliability = 'high'
    elif spread_magnitude < 1.5:
        reliability = 'medium'
    else:
        reliability = 'low'

    return {
        'class': class_name,
        'position_3d': final_position.tolist(),
        'position_std': final_std.tolist(),
        'spread_magnitude': round(float(spread_magnitude), 3),
        'num_observations': len(positions_arr),
        'num_outliers_removed': int(n_removed),
        'mean_confidence': round(float(filtered_confidences.mean()), 3),
        'reliability': reliability
    }

# Build the final scene graph
scene_graph = []

by_class_full = defaultdict(lambda: {'positions': [], 'confidences': []})
for obj in scene_objects:
    by_class_full[obj['class']]['positions'].append(obj['position_3d'])
    by_class_full[obj['class']]['confidences'].append(obj['confidence'])

for cls, data in by_class_full.items():
    clustered = cluster_object_class(data['positions'], data['confidences'], cls)
    scene_graph.append(clustered)

# Print summary sorted by reliability
for obj in sorted(scene_graph, key=lambda x: x['reliability']):
    print(f"{obj['class']:12s} | reliability: {obj['reliability']:6s} | spread: {obj['spread_magnitude']:.2f} | "
          f"obs: {obj['num_observations']:3d} | outliers removed: {obj['num_outliers_removed']}")

chair        | reliability: low    | spread: 2.96 | obs:  31 | outliers removed: 2
book         | reliability: low    | spread: 1.84 | obs: 105 | outliers removed: 12
laptop       | reliability: medium | spread: 0.78 | obs:  36 | outliers removed: 3
keyboard     | reliability: medium | spread: 0.45 | obs:   4 | outliers removed: 1
cup          | reliability: medium | spread: 0.57 | obs:  18 | outliers removed: 6
bottle       | reliability: medium | spread: 0.65 | obs:   4 | outliers removed: 1


In [12]:
scene_graph_output = {
    'objects': scene_graph,
    'relationships': []
}

for i, obj_a in enumerate(scene_graph):
    for obj_b in scene_graph[i+1:]:
        pos_a = np.array(obj_a['position_3d'])
        pos_b = np.array(obj_b['position_3d'])
        distance = float(np.linalg.norm(pos_a - pos_b))
        scene_graph_output['relationships'].append({
            'object_a': obj_a['class'],
            'object_b': obj_b['class'],
            'distance': round(distance, 3)
        })

output_path = os.path.join(base_path, 'scene_graph', 'scene_graph.json')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(scene_graph_output, f, indent=2)

print(f"Scene graph saved with {len(scene_graph)} objects and {len(scene_graph_output['relationships'])} relationships")
for rel in sorted(scene_graph_output['relationships'], key=lambda x: x['distance']):
    print(f"  {rel['object_a']:10s} <-> {rel['object_b']:10s} : {rel['distance']:.2f}")

Scene graph saved with 6 objects and 15 relationships
  cup        <-> bottle     : 1.10
  laptop     <-> book       : 1.77
  laptop     <-> bottle     : 1.94
  laptop     <-> cup        : 1.98
  book       <-> bottle     : 2.37
  laptop     <-> keyboard   : 2.71
  keyboard   <-> cup        : 3.02
  book       <-> cup        : 3.11
  keyboard   <-> bottle     : 3.85
  chair      <-> keyboard   : 4.05
  book       <-> keyboard   : 4.42
  laptop     <-> chair      : 5.19
  chair      <-> book       : 6.38
  chair      <-> cup        : 6.47
  chair      <-> bottle     : 6.97
